In [1]:
"""
MooVision Prediction Distribution Analysis
------------------------------------------
Reads all per-video prediction JSONs from a directory and produces
Altair charts saved to an HTML file.

Usage:
    python visualize_predictions.py --input_dir /path/to/prediction/jsons
                                    --output    predictions_analysis.html
"""

import json
import argparse
from pathlib import Path

import pandas as pd
import altair as alt
import sys

from pathlib import Path


from config import EVALUATION_DATA_DIR,BASELINE_MODEL_OUTPUT_DIR

In [3]:
def load_predictions(input_dir: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
        video_df  – one row per video
        event_df  – one row per event (exploded)
    """
    video_rows, event_rows = [], []

    for path in sorted(Path(input_dir).glob("*.json")):
        with open(path) as f:
            data = json.load(f)

        vid = data["identifier"]
        pen = _extract_pen(data.get("video_path", ""))
        stage = _extract_stage(data.get("video_path", ""))

        video_rows.append({
            "video":                   vid,
            "pen":                     pen,
            "weaning_stage":           stage,
            "total_duration_sec":      data.get("total_duration_sec"),
            "cross_sucking_detected":  data.get("cross_sucking_detected", False),
            "num_events":              data.get("num_events", 0),
        })

        for i, ev in enumerate(data.get("events", [])):
            event_rows.append({
                "video":           vid,
                "pen":             pen,
                "weaning_stage":   stage,
                "event_idx":       i,
                "start_sec":       ev["start_sec"],
                "end_sec":         ev["end_sec"],
                "duration_sec":    ev["duration_sec"],
                "avg_confidence":  ev["avg_confidence"],
            })

    video_df = pd.DataFrame(video_rows)
    event_df = pd.DataFrame(event_rows) if event_rows else pd.DataFrame(
        columns=["video", "pen", "weaning_stage", "event_idx",
                 "start_sec", "end_sec", "duration_sec", "avg_confidence"]
    )
    return video_df, event_df


def _extract_pen(video_path: str) -> str:
    """Best-effort pen extraction from path string."""
    import re
    m = re.search(r"Pen\s*(\d+)", video_path, re.IGNORECASE)
    return f"Pen {m.group(1)}" if m else "Unknown"


def _extract_stage(video_path: str) -> str:
    for stage in ("PREWEANING", "WEANING", "POSTWEANING"):
        if stage.lower() in video_path.lower():
            return stage
    return "Unknown"

video_df, event_df = load_predictions(BASELINE_MODEL_OUTPUT_DIR)

In [5]:
event_df

,video,pen,weaning_stage,event_idx,start_sec,end_sec,duration_sec,avg_confidence
0,ch02_20250913081207.mp4,Pen 2,PREWEANING,0,326.0,347.0,21.0,0.901
1,ch02_20250913081207.mp4,Pen 2,PREWEANING,1,624.0,639.0,15.0,0.931
2,ch02_20250913081207.mp4,Pen 2,PREWEANING,2,1125.0,1141.0,16.0,0.906
3,ch02_20250913081207.mp4,Pen 2,PREWEANING,3,1726.0,1790.0,64.0,0.905
4,ch02_20250913081207.mp4,Pen 2,PREWEANING,4,1863.0,1892.0,29.0,0.906
...,...,...,...,...,...,...,...,...
425,ch02_20251105023003.mp4,Pen 2,WEANING,33,2273.0,2423.0,150.0,0.930
426,ch02_20251105023003.mp4,Pen 2,WEANING,34,2435.0,2454.0,19.0,0.947
427,ch02_20251105023003.mp4,Pen 2,WEANING,35,2499.0,2536.0,37.0,0.924
428,ch02_20251105023003.mp4,Pen 2,WEANING,36,2547.0,2567.0,20.0,0.927
